<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/05_Dataset_and_Feature_Profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 05.1 ENVIRONMENT, CONFIGURATION AND GOOGLE DRIVE
# ============================================================

from pathlib import Path
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from scipy.stats import (
    skew,
    kurtosis,
    entropy
)

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

try:
    drive.mount(
        "/content/drive",
        force_remount=False
    )
except Exception as exc:
    print(
        f"Drive mount note: {exc}"
    )

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found:\n{PROJECT_ROOT}"
    )

# ------------------------------------------------------------
# DATASET REGISTRY
# ------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

# ------------------------------------------------------------
# TARGET REGISTRY
# ------------------------------------------------------------

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

# ------------------------------------------------------------
# RANDOM STATE
# ------------------------------------------------------------

RANDOM_SEED = 42

np.random.seed(
    RANDOM_SEED
)

print("=" * 90)
print("NOTEBOOK 05 — DATASET AND FEATURE PROFILING")
print("=" * 90)

print(
    f"Project root : {PROJECT_ROOT}"
)

print(
    f"Datasets     : {DATASET_IDS}"
)

print(
    f"Targets      : {TARGET_REGISTRY}"
)

print(
    f"Random seed  : {RANDOM_SEED}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NOTEBOOK 05 — DATASET AND FEATURE PROFILING
Project root : /content/drive/MyDrive/AIR_LLM_Research
Datasets     : ['adult_income', 'bank_marketing', 'diabetes_130us']
Targets      : {'adult_income': 'income', 'bank_marketing': 'y', 'diabetes_130us': 'readmitted'}
Random seed  : 42


In [2]:
# ============================================================
# NOTEBOOK 05.1 — ENVIRONMENT SETUP
# ============================================================

import sys
import subprocess
import importlib.util

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.1")
print("LOCAL LLM ENVIRONMENT SETUP")
print("=" * 100)


REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "torch": "torch",
    "pandas": "pandas",
    "numpy": "numpy"
}


missing_packages = []

for package_name, import_name in REQUIRED_PACKAGES.items():

    if importlib.util.find_spec(import_name) is None:
        missing_packages.append(package_name)


if missing_packages:

    print(
        "Installing missing packages:"
    )

    print(
        ", ".join(missing_packages)
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U"
        ]
        + missing_packages
    )

else:

    print(
        "All required packages are already installed."
    )


print("\nEnvironment setup complete.")

AIR-LLM — NOTEBOOK 05.1
LOCAL LLM ENVIRONMENT SETUP
All required packages are already installed.

Environment setup complete.


In [3]:
# ============================================================
# NOTEBOOK 05.2 — IMPORTS AND HARDWARE CONFIGURATION
# ============================================================

import gc
import os
import re
import json
import hashlib
import warnings

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

warnings.filterwarnings(
    "ignore"
)


print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.2")
print("IMPORTS AND HARDWARE CONFIGURATION")
print("=" * 100)


if torch.cuda.is_available():

    DEVICE = "cuda"

    GPU_NAME = torch.cuda.get_device_name(0)

    GPU_MEMORY_GB = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

else:

    DEVICE = "cpu"

    GPU_NAME = "CPU"

    GPU_MEMORY_GB = 0


print(
    f"Device       : {DEVICE}"
)

print(
    f"Hardware     : {GPU_NAME}"
)

print(
    f"GPU memory   : {GPU_MEMORY_GB:.2f} GB"
)


if DEVICE != "cuda":

    print(
        "\nWARNING: No CUDA GPU detected."
    )

    print(
        "Local LLM inference may be very slow on CPU."
    )

AIR-LLM — NOTEBOOK 05.2
IMPORTS AND HARDWARE CONFIGURATION
Device       : cpu
Hardware     : CPU
GPU memory   : 0.00 GB

Local LLM inference may be very slow on CPU.


In [4]:
# ============================================================
# NOTEBOOK 05.3 — LOCAL LLM CONFIGURATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.3")
print("LOCAL LLM CONFIGURATION")
print("=" * 100)


# ------------------------------------------------------------
# Primary local research model
# ------------------------------------------------------------

LLM_MODEL = "Qwen/Qwen3-4B-Instruct"


# ------------------------------------------------------------
# Quantization
# ------------------------------------------------------------

USE_4BIT = True

BNB_4BIT_QUANT_TYPE = "nf4"

BNB_4BIT_USE_DOUBLE_QUANT = True


if torch.cuda.is_available() and torch.cuda.is_bf16_supported():

    BNB_COMPUTE_DTYPE = torch.bfloat16

else:

    BNB_COMPUTE_DTYPE = torch.float16


# ------------------------------------------------------------
# Generation configuration
# ------------------------------------------------------------

MAX_INPUT_TOKENS = 8192

MAX_NEW_TOKENS = 1024

DO_SAMPLE = False

TEMPERATURE = None

TOP_P = None


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

if "MASTER_SEED" not in globals():

    MASTER_SEED = 42


torch.manual_seed(
    MASTER_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        MASTER_SEED
    )


print(
    f"Model                  : {LLM_MODEL}"
)

print(
    f"4-bit quantization     : {USE_4BIT}"
)

print(
    f"Quantization type      : {BNB_4BIT_QUANT_TYPE}"
)

print(
    f"Double quantization    : {BNB_4BIT_USE_DOUBLE_QUANT}"
)

print(
    f"Compute dtype          : {BNB_COMPUTE_DTYPE}"
)

print(
    f"Max input tokens       : {MAX_INPUT_TOKENS}"
)

print(
    f"Max output tokens      : {MAX_NEW_TOKENS}"
)

print(
    f"Sampling               : {DO_SAMPLE}"
)

print(
    f"Master seed            : {MASTER_SEED}"
)

AIR-LLM — NOTEBOOK 05.3
LOCAL LLM CONFIGURATION
Model                  : Qwen/Qwen3-4B-Instruct
4-bit quantization     : True
Quantization type      : nf4
Double quantization    : True
Compute dtype          : torch.float16
Max input tokens       : 8192
Max output tokens      : 1024
Sampling               : False
Master seed            : 42


In [5]:
# ============================================================
# NOTEBOOK 05.4 — SAFE LOCAL LLM INITIALIZATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.4")
print("SAFE LOCAL LLM AND TOKENIZER INITIALIZATION")
print("=" * 100)

import os
import gc
import json
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# ------------------------------------------------------------
# 1. CLEAN PREVIOUS MODEL OBJECTS
# ------------------------------------------------------------

for _name in [
    "LOCAL_LLM",
    "TOKENIZER",
    "MODEL_INPUTS",
    "GENERATED_IDS"
]:

    if _name in globals():

        try:
            del globals()[_name]
        except Exception:
            pass


gc.collect()

if torch.cuda.is_available():

    try:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception:
        pass


# ------------------------------------------------------------
# 2. RUNTIME DETECTION
# ------------------------------------------------------------

CUDA_AVAILABLE = torch.cuda.is_available()

if CUDA_AVAILABLE:

    DEVICE = "cuda"

    GPU_NAME = torch.cuda.get_device_name(0)

    GPU_MEMORY_GB = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

else:

    DEVICE = "cpu"

    GPU_NAME = "CPU"

    GPU_MEMORY_GB = 0.0


print(f"CUDA available : {CUDA_AVAILABLE}")
print(f"Device         : {DEVICE}")
print(f"Hardware       : {GPU_NAME}")

if CUDA_AVAILABLE:

    print(
        f"GPU memory     : "
        f"{GPU_MEMORY_GB:.2f} GB"
    )


# ------------------------------------------------------------
# 3. MEMORY-SAFE RESEARCH CONFIGURATION
# ------------------------------------------------------------

LLM_MODEL = (
    "Qwen/Qwen3-4B-Instruct-2507"
)

# Maximum generation length for AIR-LLM.
# Recommendations are structured JSON, so long generation
# is unnecessary.

LLM_MAX_NEW_TOKENS = 768

LLM_TEMPERATURE = 0.1

LLM_TOP_P = 0.9


# ------------------------------------------------------------
# 4. GPU SAFETY POLICY
# ------------------------------------------------------------

# Qwen3-4B is used only when a CUDA GPU is available.
#
# We intentionally DO NOT load the 4B model in FP32 on CPU.
# That configuration can consume most of the Colab RAM.

MIN_GPU_MEMORY_GB = 12.0


if CUDA_AVAILABLE:

    if GPU_MEMORY_GB >= MIN_GPU_MEMORY_GB:

        LLM_RUNTIME_MODE = (
            "GPU_4BIT"
        )

    else:

        LLM_RUNTIME_MODE = (
            "GPU_UNSUPPORTED"
        )

else:

    LLM_RUNTIME_MODE = (
        "CPU_NO_LLM"
    )


print("\nLLM runtime mode")
print("-" * 100)
print(
    f"Mode           : "
    f"{LLM_RUNTIME_MODE}"
)

print(
    f"Model          : "
    f"{LLM_MODEL}"
)

print(
    f"Max new tokens : "
    f"{LLM_MAX_NEW_TOKENS}"
)


# ------------------------------------------------------------
# 5. STOP SAFELY IF GPU IS NOT AVAILABLE
# ------------------------------------------------------------

if LLM_RUNTIME_MODE == "CPU_NO_LLM":

    print("\n" + "=" * 100)
    print("LOCAL LLM NOT LOADED")
    print("=" * 100)

    print(
        "\nNo CUDA GPU is available."
    )

    print(
        "\nThe Qwen3-4B model will NOT be loaded "
        "in CPU mode because doing so can exhaust "
        "Colab RAM."
    )

    print(
        "\nRecommended action:"
    )

    print(
        "1. Runtime"
    )

    print(
        "2. Change runtime type"
    )

    print(
        "3. Select T4 GPU"
    )

    print(
        "4. Re-run Notebook 05 from the beginning"
    )

    print(
        "\nThis is intentional and protects "
        "the experimental environment."
    )

    LOCAL_LLM = None
    TOKENIZER = None

    LLM_INITIALIZATION_STATUS = (
        "WAITING_FOR_GPU"
    )

else:

    # --------------------------------------------------------
    # 6. VALIDATE GPU MEMORY
    # --------------------------------------------------------

    if LLM_RUNTIME_MODE == "GPU_UNSUPPORTED":

        raise RuntimeError(
            "\nInsufficient GPU memory for the "
            "recommended Qwen3-4B 4-bit configuration.\n"
            f"Detected GPU memory: "
            f"{GPU_MEMORY_GB:.2f} GB\n"
            f"Required minimum: "
            f"{MIN_GPU_MEMORY_GB:.2f} GB\n\n"
            "Use a Colab T4 or better GPU."
        )


    # --------------------------------------------------------
    # 7. COMPUTE DTYPE
    # --------------------------------------------------------

    if torch.cuda.is_bf16_supported():

        COMPUTE_DTYPE = torch.bfloat16

    else:

        COMPUTE_DTYPE = torch.float16


    print(
        f"\nCompute dtype  : "
        f"{COMPUTE_DTYPE}"
    )


    # --------------------------------------------------------
    # 8. 4-BIT NF4 CONFIGURATION
    # --------------------------------------------------------

    QUANTIZATION_CONFIG = (
        BitsAndBytesConfig(

            load_in_4bit=True,

            bnb_4bit_quant_type="nf4",

            bnb_4bit_use_double_quant=True,

            bnb_4bit_compute_dtype=(
                COMPUTE_DTYPE
            ),

            bnb_4bit_quant_storage=(
                COMPUTE_DTYPE
            )

        )
    )


    print(
        "Quantization   : "
        "4-bit NF4"
    )

    print(
        "Double quant   : "
        "ENABLED"
    )


    # --------------------------------------------------------
    # 9. LOAD TOKENIZER
    # --------------------------------------------------------

    print(
        "\nLoading tokenizer..."
    )


    TOKENIZER = (
        AutoTokenizer.from_pretrained(

            LLM_MODEL,

            use_fast=True

        )
    )


    if TOKENIZER.pad_token is None:

        TOKENIZER.pad_token = (
            TOKENIZER.eos_token
        )


    print(
        "Tokenizer      : READY"
    )

    print(
        f"Tokenizer class: "
        f"{TOKENIZER.__class__.__name__}"
    )

    print(
        f"Vocabulary size: "
        f"{len(TOKENIZER):,}"
    )


    # --------------------------------------------------------
    # 10. CHAT TEMPLATE VALIDATION
    # --------------------------------------------------------

    if not hasattr(
        TOKENIZER,
        "apply_chat_template"
    ):

        raise RuntimeError(
            "Tokenizer does not provide "
            "apply_chat_template()."
        )


    if (
        TOKENIZER.chat_template
        is None
    ):

        raise RuntimeError(
            "Tokenizer has no chat template."
        )


    print(
        "Chat template  : AVAILABLE"
    )


    # --------------------------------------------------------
    # 11. LOAD 4-BIT MODEL
    # --------------------------------------------------------

    print(
        "\nLoading Qwen3-4B in 4-bit mode..."
    )

    print(
        "This may take several minutes."
    )


    LOCAL_LLM = (
        AutoModelForCausalLM.from_pretrained(

            LLM_MODEL,

            quantization_config=(
                QUANTIZATION_CONFIG
            ),

            device_map="auto",

            low_cpu_mem_usage=True,

            torch_dtype=(
                COMPUTE_DTYPE
            )

        )
    )


    # --------------------------------------------------------
    # 12. INFERENCE MODE
    # --------------------------------------------------------

    LOCAL_LLM.eval()


    for parameter in LOCAL_LLM.parameters():

        parameter.requires_grad = False


    # --------------------------------------------------------
    # 13. MODEL DEVICE
    # --------------------------------------------------------

    try:

        MODEL_DEVICE = (
            LOCAL_LLM.device
        )

    except Exception:

        MODEL_DEVICE = torch.device(
            "cuda:0"
        )


    # --------------------------------------------------------
    # 14. GENERATION CONFIGURATION
    # --------------------------------------------------------

    LOCAL_LLM_CONFIG = {

        "model":
            LLM_MODEL,

        "device":
            str(MODEL_DEVICE),

        "runtime":
            "local_gpu_4bit",

        "quantization":
            "NF4",

        "max_new_tokens":
            LLM_MAX_NEW_TOKENS,

        "temperature":
            LLM_TEMPERATURE,

        "top_p":
            LLM_TOP_P,

        "deterministic":
            True,

        "seed":
            MASTER_SEED

    }


    # --------------------------------------------------------
    # 15. FINAL VALIDATION
    # --------------------------------------------------------

    assert TOKENIZER is not None

    assert LOCAL_LLM is not None

    assert hasattr(
        TOKENIZER,
        "apply_chat_template"
    )

    assert (
        TOKENIZER.chat_template
        is not None
    )


    LLM_INITIALIZATION_STATUS = (
        "PASSED"
    )


    print(
        "\n" + "=" * 100
    )

    print(
        "LOCAL LLM INITIALIZATION : PASSED"
    )

    print(
        "=" * 100
    )

    print(
        f"Model          : {LLM_MODEL}"
    )

    print(
        f"Runtime        : "
        f"{LLM_RUNTIME_MODE}"
    )

    print(
        f"Device         : "
        f"{MODEL_DEVICE}"
    )

    print(
        "Quantization    : 4-bit NF4"
    )

    print(
        "Tokenizer       : READY"
    )

    print(
        "Chat template   : READY"
    )

    print(
        "Gradient        : DISABLED"
    )

    print(
        "Status          : READY"
    )


# ------------------------------------------------------------
# 16. CPU-SAFE STATUS
# ------------------------------------------------------------

if (
    LLM_INITIALIZATION_STATUS
    == "WAITING_FOR_GPU"
):

    print(
        "\n" + "=" * 100
    )

    print(
        "NOTEBOOK 05.4 STATUS : WAITING FOR GPU"
    )

    print(
        "=" * 100
    )

    print(
        "No large model was loaded."
    )

    print(
        "Colab RAM was protected."
    )


gc.collect()

if torch.cuda.is_available():

    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

AIR-LLM — NOTEBOOK 05.4
SAFE LOCAL LLM AND TOKENIZER INITIALIZATION
CUDA available : False
Device         : cpu
Hardware       : CPU

LLM runtime mode
----------------------------------------------------------------------------------------------------
Mode           : CPU_NO_LLM
Model          : Qwen/Qwen3-4B-Instruct-2507
Max new tokens : 768

LOCAL LLM NOT LOADED

No CUDA GPU is available.

The Qwen3-4B model will NOT be loaded in CPU mode because doing so can exhaust Colab RAM.

Recommended action:
1. Runtime
2. Change runtime type
3. Select T4 GPU
4. Re-run Notebook 05 from the beginning

This is intentional and protects the experimental environment.

NOTEBOOK 05.4 STATUS : WAITING FOR GPU
No large model was loaded.
Colab RAM was protected.


In [6]:
# ============================================================
# NOTEBOOK 05.4A — TOKENIZER VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.4A")
print("TOKENIZER VALIDATION")
print("=" * 100)

import gc
import json
import re
import time
import traceback
import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# Validate tokenizer
# ------------------------------------------------------------

if "TOKENIZER" not in globals():

    raise RuntimeError(
        "TOKENIZER is not defined. "
        "Run Cell 05.4 first."
    )


if TOKENIZER is None:

    raise RuntimeError(
        "TOKENIZER is None. "
        "Run Cell 05.4 again."
    )


if not hasattr(
    TOKENIZER,
    "apply_chat_template"
):

    raise RuntimeError(
        "Tokenizer does not support "
        "apply_chat_template()."
    )


if TOKENIZER.chat_template is None:

    raise RuntimeError(
        "Tokenizer does not contain a chat template."
    )


print(
    f"Tokenizer class : "
    f"{TOKENIZER.__class__.__name__}"
)

print(
    f"Vocabulary size : "
    f"{len(TOKENIZER):,}"
)

print(
    f"Pad token       : "
    f"{TOKENIZER.pad_token}"
)

print(
    f"EOS token       : "
    f"{TOKENIZER.eos_token}"
)


# ------------------------------------------------------------
# Chat-template test
# ------------------------------------------------------------

TEST_MESSAGES = [

    {
        "role": "system",
        "content": (
            "You are an expert researcher in "
            "missing-data imputation."
        )
    },

    {
        "role": "user",
        "content": (
            "Return a short confirmation."
        )
    }

]


TEST_PROMPT = TOKENIZER.apply_chat_template(

    TEST_MESSAGES,

    tokenize=False,

    add_generation_prompt=True

)


if not TEST_PROMPT:

    raise RuntimeError(
        "Chat-template test returned empty text."
    )


print("\nCHAT TEMPLATE TEST")
print("-" * 100)

print(
    TEST_PROMPT[:1000]
)


print("\n" + "=" * 100)
print("TOKENIZER VALIDATION : PASSED")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.4A
TOKENIZER VALIDATION


RuntimeError: TOKENIZER is None. Run Cell 05.4 again.

In [16]:
# ============================================================
# NOTEBOOK 05.5 — LOCAL LLM GENERATION CONFIGURATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.5")
print("LOCAL LLM GENERATION CONFIGURATION")
print("=" * 100)


# ------------------------------------------------------------
# Model identification
# ------------------------------------------------------------

LLM_PROVIDER = "local_huggingface"

LLM_MODEL = (
    "Qwen/Qwen3-4B-Instruct-2507"
)


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

LLM_SEED = (
    MASTER_SEED
    if "MASTER_SEED" in globals()
    else 42
)


torch.manual_seed(
    LLM_SEED
)

np.random.seed(
    LLM_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        LLM_SEED
    )


# ------------------------------------------------------------
# Generation parameters
# ------------------------------------------------------------

LLM_MAX_INPUT_TOKENS = 8192

LLM_MAX_NEW_TOKENS = 768

LLM_DO_SAMPLE = False

LLM_TEMPERATURE = 0.0

LLM_TOP_P = 1.0

LLM_REPETITION_PENALTY = 1.0


# ------------------------------------------------------------
# Runtime
# ------------------------------------------------------------

LLM_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    f"Provider             : {LLM_PROVIDER}"
)

print(
    f"Model                : {LLM_MODEL}"
)

print(
    f"Device               : {LLM_DEVICE}"
)

print(
    f"Seed                 : {LLM_SEED}"
)

print(
    f"Max input tokens     : "
    f"{LLM_MAX_INPUT_TOKENS}"
)

print(
    f"Max new tokens       : "
    f"{LLM_MAX_NEW_TOKENS}"
)

print(
    f"Deterministic        : "
    f"{not LLM_DO_SAMPLE}"
)


if LLM_DEVICE == "cpu":

    print("\nWARNING")
    print(
        "CPU mode detected."
    )
    print(
        "Use CPU only for pipeline validation."
    )
    print(
        "Use a CUDA GPU for the complete experiment."
    )


print("\n" + "=" * 100)
print("LOCAL LLM CONFIGURATION : READY")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.5
CANDIDATE IMPUTATION STRATEGY REGISTRY
Registered strategies: 7
  - mean_imputation
  - median_imputation
  - mode_imputation
  - knn_imputation
  - iterative_imputation
  - mice
  - missforest


In [17]:
# ============================================================
# NOTEBOOK 05.6 — LOCAL MODEL STATE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.6")
print("LOCAL MODEL STATE VALIDATION")
print("=" * 100)


if "LOCAL_LLM" not in globals():

    raise RuntimeError(
        "LOCAL_LLM is not defined. "
        "Run Cell 05.4 first."
    )


if LOCAL_LLM is None:

    raise RuntimeError(
        "LOCAL_LLM is None."
    )


if "TOKENIZER" not in globals():

    raise RuntimeError(
        "TOKENIZER is not defined."
    )


if TOKENIZER is None:

    raise RuntimeError(
        "TOKENIZER is None."
    )


LOCAL_LLM.eval()


print(
    f"Model class : "
    f"{LOCAL_LLM.__class__.__name__}"
)

print(
    f"Tokenizer   : "
    f"{TOKENIZER.__class__.__name__}"
)


if hasattr(
    LOCAL_LLM,
    "hf_device_map"
):

    print(
        "\nDevice map:"
    )

    print(
        LOCAL_LLM.hf_device_map
    )


print("\n" + "=" * 100)
print("LOCAL MODEL STATE : VALID")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.6
STRATEGY CAPABILITY MATRIX


,strategy_id,numeric,categorical,nonlinear,mixed_type,dependency_aware,robust_to_outliers
0,mean_imputation,True,False,False,False,False,False
1,median_imputation,True,False,False,False,False,True
2,mode_imputation,False,True,False,False,False,True
3,knn_imputation,True,False,False,False,True,False
4,iterative_imputation,True,False,False,False,True,False
5,mice,True,True,False,True,True,False
6,missforest,True,True,True,True,True,True


In [18]:
# ============================================================
# NOTEBOOK 05.7 — RECOMMENDATION SCHEMA VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.7")
print("RECOMMENDATION SCHEMA VALIDATION")
print("=" * 100)


if "RECOMMENDATION_SCHEMA" not in globals():

    raise RuntimeError(
        "RECOMMENDATION_SCHEMA is not defined."
    )


if not isinstance(
    RECOMMENDATION_SCHEMA,
    dict
):

    raise TypeError(
        "RECOMMENDATION_SCHEMA must be a dictionary."
    )


required_schema_keys = [
    "type",
    "properties",
    "required"
]


missing_schema_keys = [
    key
    for key in required_schema_keys
    if key not in RECOMMENDATION_SCHEMA
]


if missing_schema_keys:

    raise ValueError(
        "Recommendation schema is missing: "
        + ", ".join(
            missing_schema_keys
        )
    )


print(
    "Schema type:"
)

print(
    RECOMMENDATION_SCHEMA["type"]
)


print(
    "\nSchema fields:"
)

for field in (
    RECOMMENDATION_SCHEMA[
        "properties"
    ]
):

    print(
        f"  - {field}"
    )


print("\n" + "=" * 100)
print("RECOMMENDATION SCHEMA : VALID")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.7
RECOMMENDATION SCHEMA
Recommendation schema initialized.
Allowed strategies: 7


In [19]:
# ============================================================
# NOTEBOOK 05.8 — CANDIDATE STRATEGY ELIGIBILITY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.8")
print("CANDIDATE STRATEGY ELIGIBILITY")
print("=" * 100)


# ------------------------------------------------------------
# Strategy registry
# ------------------------------------------------------------

IMPUTATION_STRATEGIES = {

    "mean":

        {
            "name": "Mean Imputation",
            "type": "numeric",
            "description":
                "Replace missing numerical values "
                "with the observed mean."
        },

    "median":

        {
            "name": "Median Imputation",
            "type": "numeric",
            "description":
                "Replace missing numerical values "
                "with the observed median."
        },

    "knn":

        {
            "name": "KNN Imputation",
            "type": "numeric",
            "description":
                "Use neighboring observations to "
                "estimate missing numerical values."
        },

    "iterative":

        {
            "name": "Iterative Imputation",
            "type": "numeric",
            "description":
                "Iteratively model each incomplete "
                "feature using other observed features."
        },

    "mice":

        {
            "name": "MICE",
            "type": "numeric",
            "description":
                "Multiple imputation by chained equations."
        },

    "missforest":

        {
            "name": "MissForest",
            "type": "mixed",
            "description":
                "Random-forest-based iterative imputation "
                "for nonlinear relationships and mixed data."
        },

    "mode":

        {
            "name": "Mode Imputation",
            "type": "categorical",
            "description":
                "Replace missing categorical values "
                "with the most frequent category."
        },

    "constant":

        {
            "name": "Constant Imputation",
            "type": "mixed",
            "description":
                "Replace missing values with a predefined "
                "constant or missingness category."
        }

}


def get_feature_type(
    dataset_id,
    feature
):

    row = FEATURE_PROFILE_DF[
        (
            FEATURE_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        )
        &
        (
            FEATURE_PROFILE_DF[
                "feature"
            ] == feature
        )
    ]


    if row.empty:

        raise ValueError(
            f"Feature not found: "
            f"{dataset_id} / {feature}"
        )


    numeric_columns = [

        "is_numeric",

        "numeric"

    ]


    for column in numeric_columns:

        if column in row.columns:

            value = row.iloc[0][column]

            if pd.notna(value):

                if bool(value):

                    return "numeric"


    return "categorical"


def get_candidate_strategies(
    dataset_id,
    feature
):

    feature_type = get_feature_type(
        dataset_id,
        feature
    )


    candidates = []


    for strategy_id, metadata in (
        IMPUTATION_STRATEGIES.items()
    ):

        strategy_type = metadata["type"]


        if strategy_type == "mixed":

            candidates.append(
                strategy_id
            )


        elif strategy_type == feature_type:

            candidates.append(
                strategy_id
            )


    return candidates


print(
    f"Registered strategies: "
    f"{len(IMPUTATION_STRATEGIES)}"
)


print("\n" + "=" * 100)
print("CANDIDATE STRATEGY REGISTRY : READY")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.8
LOCAL LLM SYSTEM PROMPT
System prompt initialized.


In [20]:
# ============================================================
# NOTEBOOK 05.9 — LOCAL LLM PROMPT BUILDER
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.9")
print("LOCAL LLM PROMPT BUILDER")
print("=" * 100)


if "SYSTEM_PROMPT" not in globals():

    raise RuntimeError(
        "SYSTEM_PROMPT is not defined."
    )


def build_local_system_prompt():

    return SYSTEM_PROMPT + """

IMPORTANT AIR-LLM CONSTRAINTS:

1. Recommend only strategies listed as eligible.
2. Do not invent strategy IDs.
3. Rank strategies using the supplied empirical evidence.
4. Consider missingness, distribution, dependency,
   mutual information, and task relevance.
5. Return valid JSON only.
6. The selected strategy MUST be one of the eligible strategies.
7. Confidence must be between 0 and 1.
8. Provide evidence-based reasoning.
9. Do not claim experimental results that were not supplied.
"""


LOCAL_SYSTEM_PROMPT = (
    build_local_system_prompt()
)


def build_local_user_prompt(
    context
):

    candidate_strategies = (
        context.get(
            "candidate_strategies",
            []
        )
    )


    context_copy = dict(
        context
    )


    context_copy[
        "candidate_strategies"
    ] = candidate_strategies


    return (
        "Use the following AIR-LLM feature evidence "
        "to recommend an imputation strategy.\n\n"
        "FEATURE EVIDENCE:\n"
        + json.dumps(
            context_copy,
            indent=2,
            default=str
        )
        + "\n\n"
        "Return ONLY the required JSON object."
    )


print(
    "Local prompt builder initialized."
)

print("\n" + "=" * 100)
print("PROMPT BUILDER : READY")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.9
FEATURE CONTEXT BUILDER
Context builder initialized.


In [21]:
# ============================================================
# NOTEBOOK 05.10 — FEATURE CONTEXT ENRICHMENT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.10")
print("FEATURE CONTEXT ENRICHMENT")
print("=" * 100)


def prepare_recommendation_context(
    context
):

    context = dict(
        context
    )


    dataset_id = context[
        "dataset_id"
    ]

    feature = context[
        "feature"
    ]


    candidates = get_candidate_strategies(

        dataset_id,

        feature

    )


    context[
        "candidate_strategies"
    ] = candidates


    context[
        "strategy_metadata"
    ] = {

        strategy_id:
            IMPUTATION_STRATEGIES[
                strategy_id
            ]

        for strategy_id
        in candidates

    }


    return context


print(
    "Feature context enrichment function : READY"
)

print("\n" + "=" * 100)
print("FEATURE CONTEXT ENRICHMENT : READY")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.10
USER PROMPT BUILDER
User prompt builder initialized.


In [22]:
# ============================================================
# NOTEBOOK 05.11 — LOCAL LLM GENERATION ENGINE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.11")
print("LOCAL LLM GENERATION ENGINE")
print("=" * 100)


def generate_local_response(
    user_prompt
):

    # --------------------------------------------------------
    # Validate model
    # --------------------------------------------------------

    if TOKENIZER is None:

        raise RuntimeError(
            "TOKENIZER is None."
        )


    if LOCAL_LLM is None:

        raise RuntimeError(
            "LOCAL_LLM is None."
        )


    # --------------------------------------------------------
    # Messages
    # --------------------------------------------------------

    messages = [

        {
            "role": "system",
            "content": LOCAL_SYSTEM_PROMPT
        },

        {
            "role": "user",
            "content": user_prompt
        }

    ]


    # --------------------------------------------------------
    # Apply Qwen chat template
    # --------------------------------------------------------

    model_inputs = TOKENIZER.apply_chat_template(

        messages,

        tokenize=True,

        add_generation_prompt=True,

        return_dict=True,

        return_tensors="pt"

    )


    # --------------------------------------------------------
    # Truncate if required
    # --------------------------------------------------------

    if (
        model_inputs[
            "input_ids"
        ].shape[-1]
        >
        LLM_MAX_INPUT_TOKENS
    ):

        input_ids = model_inputs[
            "input_ids"
        ][:, -LLM_MAX_INPUT_TOKENS:]


        if "attention_mask" in model_inputs:

            attention_mask = model_inputs[
                "attention_mask"
            ][:, -LLM_MAX_INPUT_TOKENS:]

        else:

            attention_mask = None


        model_inputs = {
            "input_ids":
                input_ids
        }


        if attention_mask is not None:

            model_inputs[
                "attention_mask"
            ] = attention_mask


    # --------------------------------------------------------
    # Move to model device
    # --------------------------------------------------------

    if hasattr(
        LOCAL_LLM,
        "device"
    ):

        model_device = (
            LOCAL_LLM.device
        )

    else:

        model_device = torch.device(
            LLM_DEVICE
        )


    model_inputs = {

        key:
            value.to(
                model_device
            )

        for key, value
        in model_inputs.items()

    }


    # --------------------------------------------------------
    # Generation
    # --------------------------------------------------------

    generation_kwargs = {

        "max_new_tokens":
            LLM_MAX_NEW_TOKENS,

        "do_sample":
            LLM_DO_SAMPLE,

        "pad_token_id":
            TOKENIZER.pad_token_id,

        "eos_token_id":
            TOKENIZER.eos_token_id

    }


    if LLM_DO_SAMPLE:

        generation_kwargs[
            "temperature"
        ] = LLM_TEMPERATURE

        generation_kwargs[
            "top_p"
        ] = LLM_TOP_P


    with torch.inference_mode():

        generated_ids = (
            LOCAL_LLM.generate(

                **model_inputs,

                **generation_kwargs

            )
        )


    # --------------------------------------------------------
    # Remove input prompt
    # --------------------------------------------------------

    input_length = (
        model_inputs[
            "input_ids"
        ].shape[-1]
    )


    output_ids = generated_ids[
        0,
        input_length:
    ]


    # --------------------------------------------------------
    # Decode
    # --------------------------------------------------------

    raw_text = TOKENIZER.decode(

        output_ids,

        skip_special_tokens=True

    ).strip()


    if not raw_text:

        raise RuntimeError(
            "Local LLM generated an empty response."
        )


    return raw_text


print(
    "Local generation engine : READY"
)

print("\n" + "=" * 100)
print("LOCAL GENERATION ENGINE : READY")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.11
LOCAL LLM GENERATION ENGINE
Local LLM generation engine initialized.


In [23]:
# ============================================================
# NOTEBOOK 05.12 — JSON EXTRACTION AND VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.12")
print("JSON EXTRACTION AND VALIDATION")
print("=" * 100)


def extract_json_object(
    raw_text
):

    if not raw_text:

        raise ValueError(
            "Empty LLM response."
        )


    text = raw_text.strip()


    # --------------------------------------------------------
    # Remove markdown fences
    # --------------------------------------------------------

    text = re.sub(

        r"^```(?:json)?",

        "",

        text,

        flags=re.IGNORECASE

    )

    text = re.sub(

        r"```$",

        "",

        text

    )


    text = text.strip()


    # --------------------------------------------------------
    # Direct JSON
    # --------------------------------------------------------

    try:

        parsed = json.loads(
            text
        )

        if isinstance(
            parsed,
            dict
        ):

            return parsed

    except Exception:

        pass


    # --------------------------------------------------------
    # Extract first JSON object
    # --------------------------------------------------------

    start = text.find(
        "{"
    )

    end = text.rfind(
        "}"
    )


    if (
        start == -1
        or
        end == -1
        or
        end <= start
    ):

        raise ValueError(
            "No JSON object found in LLM response."
        )


    candidate = text[
        start:
        end + 1
    ]


    try:

        parsed = json.loads(
            candidate
        )

    except Exception as exc:

        raise ValueError(
            "Unable to parse JSON from LLM response."
        ) from exc


    if not isinstance(
        parsed,
        dict
    ):

        raise ValueError(
            "Extracted JSON is not an object."
        )


    return parsed


def validate_recommendation(
    result,
    context
):

    errors = []


    # --------------------------------------------------------
    # Required top-level fields
    # --------------------------------------------------------

    required_fields = [

        "selected_strategy",

        "recommendations",

        "overall_reasoning"

    ]


    for field in required_fields:

        if field not in result:

            errors.append(
                f"Missing field: {field}"
            )


    if errors:

        return False, errors


    # --------------------------------------------------------
    # Candidate strategies
    # --------------------------------------------------------

    eligible = set(
        context.get(
            "candidate_strategies",
            []
        )
    )


    selected_strategy = result[
        "selected_strategy"
    ]


    if selected_strategy not in eligible:

        errors.append(
            "Selected strategy "
            f"'{selected_strategy}' "
            "is not in eligible candidate strategies."
        )


    # --------------------------------------------------------
    # Recommendations
    # --------------------------------------------------------

    recommendations = result[
        "recommendations"
    ]


    if not isinstance(
        recommendations,
        list
    ):

        errors.append(
            "'recommendations' must be a list."
        )

        return False, errors


    if len(
        recommendations
    ) == 0:

        errors.append(
            "Recommendations list is empty."
        )


    seen_ranks = set()


    for recommendation in recommendations:

        if not isinstance(
            recommendation,
            dict
        ):

            errors.append(
                "Each recommendation must be an object."
            )

            continue


        for field in [

            "rank",

            "strategy_id",

            "confidence",

            "rationale"

        ]:

            if field not in recommendation:

                errors.append(
                    f"Recommendation missing "
                    f"'{field}'."
                )


        if "rank" in recommendation:

            try:

                rank = int(
                    recommendation[
                        "rank"
                    ]
                )

                if rank in seen_ranks:

                    errors.append(
                        f"Duplicate rank: {rank}"
                    )

                seen_ranks.add(
                    rank
                )

            except Exception:

                errors.append(
                    "Invalid recommendation rank."
                )


        if "strategy_id" in recommendation:

            strategy_id = (
                recommendation[
                    "strategy_id"
                ]
            )


            if strategy_id not in eligible:

                errors.append(
                    f"Strategy '{strategy_id}' "
                    "is not eligible."
                )


        if "confidence" in recommendation:

            try:

                confidence = float(
                    recommendation[
                        "confidence"
                    ]
                )


                if not (
                    0.0
                    <=
                    confidence
                    <=
                    1.0
                ):

                    errors.append(
                        "Confidence must be "
                        "between 0 and 1."
                    )

            except Exception:

                errors.append(
                    "Invalid confidence value."
                )


        if "rationale" in recommendation:

            if not str(
                recommendation[
                    "rationale"
                ]
            ).strip():

                errors.append(
                    "Recommendation rationale "
                    "cannot be empty."
                )


    # --------------------------------------------------------
    # Overall reasoning
    # --------------------------------------------------------

    if not str(
        result[
            "overall_reasoning"
        ]
    ).strip():

        errors.append(
            "Overall reasoning cannot be empty."
        )


    return (
        len(errors) == 0,
        errors
    )


print(
    "JSON extraction function : READY"
)

print(
    "Recommendation validator  : READY"
)

print("\n" + "=" * 100)
print("JSON VALIDATION LAYER : READY")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.12
RECOMMENDATION PARSING AND VALIDATION
Recommendation validation engine initialized.


In [24]:
# ============================================================
# NOTEBOOK 05.12A — LOCAL LLM SMOKE TEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.12A")
print("LOCAL LLM SMOKE TEST")
print("=" * 100)


SMOKE_MESSAGES = [

    {
        "role": "system",
        "content": (
            "You are a research assistant. "
            "Return valid JSON only."
        )
    },

    {
        "role": "user",
        "content": (
            'Return exactly this JSON object: '
            '{"status":"success"}'
        )
    }

]


SMOKE_INPUT = TOKENIZER.apply_chat_template(

    SMOKE_MESSAGES,

    tokenize=True,

    add_generation_prompt=True,

    return_dict=True,

    return_tensors="pt"

)


SMOKE_INPUT = {

    key:
        value.to(
            LOCAL_LLM.device
        )

    for key, value
    in SMOKE_INPUT.items()

}


with torch.inference_mode():

    SMOKE_OUTPUT = LOCAL_LLM.generate(

        **SMOKE_INPUT,

        max_new_tokens=50,

        do_sample=False,

        pad_token_id=
            TOKENIZER.pad_token_id,

        eos_token_id=
            TOKENIZER.eos_token_id

    )


SMOKE_INPUT_LENGTH = (
    SMOKE_INPUT[
        "input_ids"
    ].shape[-1]
)


SMOKE_GENERATED = (
    SMOKE_OUTPUT[
        0,
        SMOKE_INPUT_LENGTH:
    ]
)


SMOKE_RESPONSE = TOKENIZER.decode(

    SMOKE_GENERATED,

    skip_special_tokens=True

).strip()


print("\nMODEL RESPONSE")
print("-" * 100)

print(
    SMOKE_RESPONSE
)


if not SMOKE_RESPONSE:

    raise RuntimeError(
        "Local model returned empty output."
    )


print("\n" + "=" * 100)
print("LOCAL LLM SMOKE TEST : PASSED")
print("=" * 100)

AIR-LLM — NOTEBOOK 05.13
SINGLE LOCAL LLM RECOMMENDATION TEST


AttributeError: 'NoneType' object has no attribute 'apply_chat_template'

In [ ]:
# ============================================================
# NOTEBOOK 05.13 — SINGLE LOCAL LLM RECOMMENDATION TEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.13")
print("SINGLE LOCAL LLM RECOMMENDATION TEST")
print("=" * 100)


def request_recommendation(
    context
):

    # --------------------------------------------------------
    # Enrich context
    # --------------------------------------------------------

    context = prepare_recommendation_context(
        context
    )


    # --------------------------------------------------------
    # Build prompt
    # --------------------------------------------------------

    user_prompt = build_local_user_prompt(
        context
    )


    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    raw_text = generate_local_response(
        user_prompt
    )


    # --------------------------------------------------------
    # Parse
    # --------------------------------------------------------

    result = extract_json_object(
        raw_text
    )


    # --------------------------------------------------------
    # Validate
    # --------------------------------------------------------

    valid, errors = validate_recommendation(

        result,

        context

    )


    if not valid:

        raise ValueError(

            "Invalid AIR-LLM recommendation:\n"
            +
            "\n".join(
                f"  - {error}"
                for error in errors
            )
            +
            "\n\nRAW RESPONSE:\n"
            +
            raw_text

        )


    return (
        result,
        raw_text,
        context
    )


# ------------------------------------------------------------
# Deterministic test feature
# ------------------------------------------------------------

first_dataset = DATASETS[0]


feature_candidates = [

    feature

    for feature
    in FEATURE_PROFILE_DF[
        FEATURE_PROFILE_DF[
            "dataset_id"
        ] == first_dataset
    ][
        "feature"
    ].tolist()

    if feature !=
    TARGET_REGISTRY[
        first_dataset
    ]

]


if not feature_candidates:

    raise RuntimeError(
        "No feature available for test."
    )


first_feature = (
    feature_candidates[0]
)


TEST_CONTEXT_BASE = build_feature_context(

    first_dataset,

    first_feature

)


TEST_RESULT, TEST_RAW_RESPONSE, TEST_CONTEXT = (
    request_recommendation(
        TEST_CONTEXT_BASE
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nTEST FEATURE")
print("-" * 100)

print(
    f"Dataset : "
    f"{TEST_CONTEXT['dataset_id']}"
)

print(
    f"Feature : "
    f"{TEST_CONTEXT['feature']}"
)


print(
    "\nEligible strategies:"
)

print(
    TEST_CONTEXT[
        "candidate_strategies"
    ]
)


print("\nSELECTED STRATEGY")
print("-" * 100)

print(
    TEST_RESULT[
        "selected_strategy"
    ]
)


print("\nRECOMMENDATIONS")
print("-" * 100)


for recommendation in (
    TEST_RESULT[
        "recommendations"
    ]
):

    print(

        f"{recommendation['rank']}. "
        f"{recommendation['strategy_id']} "
        f"| confidence="
        f"{float(recommendation['confidence']):.3f}"

    )

    print(
        f"   {recommendation['rationale']}"
    )


print("\nOVERALL REASONING")
print("-" * 100)

print(
    TEST_RESULT[
        "overall_reasoning"
    ]
)


print("\n" + "=" * 100)
print("SINGLE LOCAL LLM REQUEST : PASSED")
print("=" * 100)

COMPUTATIONAL SCALE PROFILE COMPLETED


,dataset_id,rows,features,numeric_features,categorical_features,cells,missing_cells,missing_cell_rate,memory_mb,scale_category,row_scale_category,feature_scale_category
0,adult_income,19522,14,6,8,273308,0,0.00000,10.580482,small,moderate,low
1,bank_marketing,27126,16,7,9,434016,0,0.00000,15.448343,small,moderate,low
2,diabetes_130us,61059,47,11,36,2869773,224503,0.07823,112.802345,medium,moderate,moderate


In [ ]:
# ============================================================
# NOTEBOOK 05.14 — RECOMMENDATION RESULT NORMALIZATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.14")
print("RECOMMENDATION RESULT NORMALIZATION")
print("=" * 100)


def normalize_recommendation(
    result,
    context
):

    rows = []


    selected_strategy = (
        result[
            "selected_strategy"
        ]
    )


    for recommendation in (
        result[
            "recommendations"
        ]
    ):

        rows.append({

            "dataset_id":
                context[
                    "dataset_id"
                ],

            "feature":
                context[
                    "feature"
                ],

            "selected_strategy":
                selected_strategy,

            "rank":
                int(
                    recommendation[
                        "rank"
                    ]
                ),

            "strategy_id":
                recommendation[
                    "strategy_id"
                ],

            "confidence":
                float(
                    recommendation[
                        "confidence"
                    ]
                ),

            "rationale":
                recommendation[
                    "rationale"
                ],

            "overall_reasoning":
                result[
                    "overall_reasoning"
                ]

        })


    return rows


TEST_RECOMMENDATION_ROWS = (
    normalize_recommendation(

        TEST_RESULT,

        TEST_CONTEXT

    )
)


TEST_RECOMMENDATION_DF = pd.DataFrame(
    TEST_RECOMMENDATION_ROWS
)


display(
    TEST_RECOMMENDATION_DF
)


print("\n" + "=" * 100)
print("RECOMMENDATION NORMALIZATION : PASSED")
print("=" * 100)

DATASET PROFILES CONSTRUCTED
adult_income         | Rows=19,522 | Features=14 | Numeric=6 | Categorical=8 | Incomplete=0
bank_marketing       | Rows=27,126 | Features=16 | Numeric=7 | Categorical=9 | Incomplete=0
diabetes_130us       | Rows=61,059 | Features=47 | Numeric=11 | Categorical=36 | Incomplete=9


In [ ]:
# ============================================================
# NOTEBOOK 05.15 — BATCH LOCAL LLM RECOMMENDATIONS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.15")
print("BATCH LOCAL LLM RECOMMENDATIONS")
print("=" * 100)


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

if "RECOMMENDATION_DIR" not in globals():

    RECOMMENDATION_DIR = (
        PROJECT_ROOT
        / "outputs"
        / "recommendations"
    )


RECOMMENDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


BATCH_OUTPUT_PATH = (
    RECOMMENDATION_DIR
    / "air_llm_recommendations.csv"
)


BATCH_FAILURE_PATH = (
    RECOMMENDATION_DIR
    / "air_llm_failures.csv"
)


# ------------------------------------------------------------
# Resume previous results
# ------------------------------------------------------------

if BATCH_OUTPUT_PATH.exists():

    RESULTS_DF = pd.read_csv(
        BATCH_OUTPUT_PATH
    )

else:

    RESULTS_DF = pd.DataFrame()


if BATCH_FAILURE_PATH.exists():

    FAILURES_DF = pd.read_csv(
        BATCH_FAILURE_PATH
    )

else:

    FAILURES_DF = pd.DataFrame()


# ------------------------------------------------------------
# Build feature list
# ------------------------------------------------------------

FEATURE_TASKS = []


for dataset_id in DATASETS:

    target = TARGET_REGISTRY[
        dataset_id
    ]


    dataset_features = (

        FEATURE_PROFILE_DF[

            FEATURE_PROFILE_DF[
                "dataset_id"
            ]
            ==
            dataset_id

        ][
            "feature"
        ]

        .dropna()

        .astype(str)

        .tolist()

    )


    for feature in dataset_features:

        if feature == target:

            continue


        FEATURE_TASKS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature

        })


print(
    f"Total feature tasks : "
    f"{len(FEATURE_TASKS):,}"
)


# ------------------------------------------------------------
# Resume logic
# ------------------------------------------------------------

completed_keys = set()


if not RESULTS_DF.empty:

    for _, row in RESULTS_DF.iterrows():

        completed_keys.add(

            (
                str(
                    row[
                        "dataset_id"
                    ]
                ),

                str(
                    row[
                        "feature"
                    ]
                )

            )

        )


print(
    f"Already completed   : "
    f"{len(completed_keys):,}"
)


# ------------------------------------------------------------
# Batch execution
# ------------------------------------------------------------

new_rows = []


for task_number, task in enumerate(

    FEATURE_TASKS,

    start=1

):

    dataset_id = task[
        "dataset_id"
    ]

    feature = task[
        "feature"
    ]


    key = (
        dataset_id,
        feature
    )


    if key in completed_keys:

        continue


    print("\n" + "=" * 100)

    print(
        f"TASK {task_number}/"
        f"{len(FEATURE_TASKS)}"
    )

    print(
        f"Dataset : {dataset_id}"
    )

    print(
        f"Feature : {feature}"
    )


    try:

        context = build_feature_context(

            dataset_id,

            feature

        )


        result, raw_text, context = (
            request_recommendation(
                context
            )
        )


        rows = normalize_recommendation(

            result,

            context

        )


        new_rows.extend(
            rows
        )


        completed_keys.add(
            key
        )


        # ----------------------------------------------------
        # Incremental checkpoint
        # ----------------------------------------------------

        checkpoint_df = pd.DataFrame(
            new_rows
        )


        if not RESULTS_DF.empty:

            checkpoint_df = pd.concat(

                [
                    RESULTS_DF,
                    checkpoint_df
                ],

                ignore_index=True

            )


        checkpoint_df.to_csv(

            BATCH_OUTPUT_PATH,

            index=False

        )


        print(
            f"Selected strategy: "
            f"{result['selected_strategy']}"
        )

        print(
            "STATUS : SUCCESS"
        )


    except Exception as exc:

        failure_row = {

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "error_type":
                type(
                    exc
                ).__name__,

            "error_message":
                str(
                    exc
                ),

            "timestamp":
                pd.Timestamp.utcnow().isoformat()

        }


        FAILURES_DF = pd.concat(

            [

                FAILURES_DF,

                pd.DataFrame(
                    [failure_row]
                )

            ],

            ignore_index=True

        )


        FAILURES_DF.to_csv(

            BATCH_FAILURE_PATH,

            index=False

        )


        print(
            "STATUS : FAILED"
        )

        print(
            f"Error: {exc}"
        )


    # --------------------------------------------------------
    # Memory cleanup
    # --------------------------------------------------------

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ------------------------------------------------------------
# Final combined results
# ------------------------------------------------------------

if BATCH_OUTPUT_PATH.exists():

    AIR_LLM_RECOMMENDATIONS_DF = pd.read_csv(

        BATCH_OUTPUT_PATH

    )

else:

    AIR_LLM_RECOMMENDATIONS_DF = (
        pd.DataFrame()
    )


print("\n" + "=" * 100)

print(
    "BATCH RECOMMENDATION PROCESS COMPLETED"
)

print(
    f"Recommendation rows : "
    f"{len(AIR_LLM_RECOMMENDATIONS_DF):,}"
)

print(
    f"Failures             : "
    f"{len(FAILURES_DF):,}"
)

print(
    f"Output               : "
    f"{BATCH_OUTPUT_PATH}"
)

print("=" * 100)

NOTEBOOK 05 — PROFILE OBJECT AVAILABILITY
TRAINING_DATA                  : AVAILABLE
TARGET_REGISTRY                : AVAILABLE
DATASET_SIZE_DF                : AVAILABLE
FEATURE_COUNT_DF               : AVAILABLE
TYPE_RATIO_DF                  : AVAILABLE
CLASS_DISTRIBUTION_DF          : AVAILABLE
MISSINGNESS_DF                 : AVAILABLE
CARDINALITY_DF                 : AVAILABLE
DISTRIBUTION_DF                : AVAILABLE
OUTLIER_DF                     : AVAILABLE
CORRELATION_PROFILE            : AVAILABLE
MUTUAL_INFORMATION_DF          : AVAILABLE
DEPENDENCY_DF                  : AVAILABLE
COMPUTATIONAL_SCALE_DF         : AVAILABLE

All Notebook 05 profile objects are available.


In [ ]:
# ============================================================
# NOTEBOOK 05.16 — RECOMMENDATION QUALITY VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.16")
print("RECOMMENDATION QUALITY VALIDATION")
print("=" * 100)


if AIR_LLM_RECOMMENDATIONS_DF.empty:

    raise RuntimeError(
        "No AIR-LLM recommendations available."
    )


VALIDATION_ROWS = []


for _, row in AIR_LLM_RECOMMENDATIONS_DF.iterrows():

    dataset_id = row[
        "dataset_id"
    ]

    feature = row[
        "feature"
    ]

    strategy_id = row[
        "strategy_id"
    ]


    try:

        eligible = (
            get_candidate_strategies(

                dataset_id,

                feature

            )
        )


        strategy_valid = (
            strategy_id
            in
            eligible
        )


    except Exception:

        eligible = []

        strategy_valid = False


    confidence = float(
        row[
            "confidence"
        ]
    )


    confidence_valid = (
        0.0
        <=
        confidence
        <=
        1.0
    )


    rationale_valid = (
        isinstance(
            row[
                "rationale"
            ],
            str
        )
        and
        len(
            row[
                "rationale"
            ].strip()
        )
        > 0
    )


    VALIDATION_ROWS.append({

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "strategy_id":
            strategy_id,

        "strategy_valid":
            strategy_valid,

        "confidence_valid":
            confidence_valid,

        "rationale_valid":
            rationale_valid,

        "overall_valid":
            (
                strategy_valid
                and
                confidence_valid
                and
                rationale_valid
            )

    })


RECOMMENDATION_VALIDATION_DF = pd.DataFrame(
    VALIDATION_ROWS
)


display(
    RECOMMENDATION_VALIDATION_DF.head(20)
)


valid_count = int(
    RECOMMENDATION_VALIDATION_DF[
        "overall_valid"
    ].sum()
)


total_count = len(
    RECOMMENDATION_VALIDATION_DF
)


valid_rate = (
    valid_count / total_count
    if total_count > 0
    else 0
)


print(
    f"\nValid records : "
    f"{valid_count:,}/{total_count:,}"
)

print(
    f"Validation rate: "
    f"{valid_rate:.2%}"
)


print("\n" + "=" * 100)
print("RECOMMENDATION QUALITY VALIDATION : COMPLETED")
print("=" * 100)

FEATURE TYPE PROFILE
Rows : 80
Datasets : 3
Numerical : 24
Categorical : 53
Targets : 3


,dataset_id,feature,feature_role,dtype,is_numeric,is_categorical,is_target
0,adult_income,age,numerical,int64,True,False,False
1,adult_income,workclass,categorical,object,False,True,False
2,adult_income,fnlwgt,numerical,int64,True,False,False
3,adult_income,education,categorical,object,False,True,False
4,adult_income,education_num,numerical,int64,True,False,False
5,adult_income,marital_status,categorical,object,False,True,False
6,adult_income,occupation,categorical,object,False,True,False
7,adult_income,relationship,categorical,object,False,True,False
8,adult_income,race,categorical,object,False,True,False
9,adult_income,sex,categorical,object,False,True,False


In [ ]:
# ============================================================
# NOTEBOOK 05.17 — SELECTED STRATEGY TABLE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.17")
print("SELECTED STRATEGY TABLE")
print("=" * 100)


SELECTED_STRATEGIES_DF = (

    AIR_LLM_RECOMMENDATIONS_DF[

        AIR_LLM_RECOMMENDATIONS_DF[
            "rank"
        ] == 1

    ]

    .copy()

)


SELECTED_STRATEGIES_DF = (

    SELECTED_STRATEGIES_DF[

        [
            "dataset_id",

            "feature",

            "selected_strategy",

            "confidence",

            "rationale",

            "overall_reasoning"

        ]

    ]

    .drop_duplicates(

        subset=[
            "dataset_id",
            "feature"
        ]

    )

    .reset_index(drop=True)

)


display(
    SELECTED_STRATEGIES_DF.head(20)
)


print(
    f"Selected strategies: "
    f"{len(SELECTED_STRATEGIES_DF):,}"
)


print("\n" + "=" * 100)
print("SELECTED STRATEGY TABLE : READY")
print("=" * 100)

FEATURE MISSINGNESS PROFILE
Rows       : 80
Incomplete : 9
Complete   : 71
Targets    : 3


,dataset_id,feature,missing_count,missing_rate,missingness_category,is_target
0,adult_income,age,0,0.0,none,False
1,adult_income,workclass,0,0.0,none,False
2,adult_income,fnlwgt,0,0.0,none,False
3,adult_income,education,0,0.0,none,False
4,adult_income,education_num,0,0.0,none,False
5,adult_income,marital_status,0,0.0,none,False
6,adult_income,occupation,0,0.0,none,False
7,adult_income,relationship,0,0.0,none,False
8,adult_income,race,0,0.0,none,False
9,adult_income,sex,0,0.0,none,False


In [ ]:
# ============================================================
# NOTEBOOK 05.18 — RECOMMENDATION DISTRIBUTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.18")
print("RECOMMENDATION DISTRIBUTION")
print("=" * 100)


STRATEGY_DISTRIBUTION_DF = (

    SELECTED_STRATEGIES_DF[

        "selected_strategy"

    ]

    .value_counts()

    .rename_axis(
        "strategy_id"
    )

    .reset_index(
        name="count"
    )

)


STRATEGY_DISTRIBUTION_DF[
    "percentage"
] = (

    STRATEGY_DISTRIBUTION_DF[
        "count"
    ]
    /
    len(
        SELECTED_STRATEGIES_DF
    )
    *
    100

)


display(
    STRATEGY_DISTRIBUTION_DF
)


print("\n" + "=" * 100)
print("RECOMMENDATION DISTRIBUTION : COMPLETED")
print("=" * 100)

FEATURE DISTRIBUTION PROFILE
Total features : 77
Numerical      : 24
Categorical    : 53


,dataset_id,feature,distribution_type,mean,median,std,min,max,skewness,kurtosis,distribution_shape,top_category_frequency
0,adult_income,age,numerical,38.539852,37.0,13.645484,17.0,90.0,0.558961,-0.164794,approximately_symmetric,NaN
1,adult_income,workclass,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.693576
2,adult_income,fnlwgt,numerical,190020.295205,178319.0,106163.578269,12285.0,1484705.0,1.430776,5.829256,right_skewed,NaN
3,adult_income,education,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.323328
4,adult_income,education_num,numerical,10.083752,10.0,2.567326,1.0,16.0,-0.297397,0.588060,approximately_symmetric,NaN
5,adult_income,marital_status,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.459379
6,adult_income,occupation,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.127753
7,adult_income,relationship,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.404979
8,adult_income,race,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.856879
9,adult_income,sex,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.668528


In [ ]:
# ============================================================
# NOTEBOOK 05.19 — SAVE FINAL RECOMMENDATIONS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.19")
print("SAVE FINAL RECOMMENDATIONS")
print("=" * 100)


if "RECOMMENDATION_DIR" not in globals():

    RECOMMENDATION_DIR = (

        PROJECT_ROOT
        / "outputs"
        / "recommendations"

    )


RECOMMENDATION_DIR.mkdir(

    parents=True,

    exist_ok=True

)


SELECTED_PATH = (

    RECOMMENDATION_DIR
    / "selected_strategies.csv"

)


DISTRIBUTION_PATH = (

    RECOMMENDATION_DIR
    / "strategy_distribution.csv"

)


VALIDATION_PATH = (

    RECOMMENDATION_DIR
    / "recommendation_validation.csv"

)


FULL_RECOMMENDATION_PATH = (

    RECOMMENDATION_DIR
    / "air_llm_recommendations.csv"

)


SELECTED_STRATEGIES_DF.to_csv(

    SELECTED_PATH,

    index=False

)


STRATEGY_DISTRIBUTION_DF.to_csv(

    DISTRIBUTION_PATH,

    index=False

)


RECOMMENDATION_VALIDATION_DF.to_csv(

    VALIDATION_PATH,

    index=False

)


AIR_LLM_RECOMMENDATIONS_DF.to_csv(

    FULL_RECOMMENDATION_PATH,

    index=False

)


print(
    "Saved:"
)


for path in [

    FULL_RECOMMENDATION_PATH,

    SELECTED_PATH,

    DISTRIBUTION_PATH,

    VALIDATION_PATH

]:

    print(
        f"  {path}"
    )


print("\n" + "=" * 100)
print("RECOMMENDATION OUTPUTS : SAVED")
print("=" * 100)

FEATURE DEPENDENCY PROFILE
Features                 : 77
Strong dependencies     : 2
Moderate dependencies   : 2
Weak dependencies       : 8
Predictive dependencies : 69


,dataset_id,feature,max_abs_spearman,strongest_correlated_feature,mutual_information,dependency_strength,has_correlated_predictor,has_predictive_dependency
0,adult_income,relationship,0.000000,None,0.115568,very_weak,False,True
1,adult_income,marital_status,0.000000,None,0.107862,very_weak,False,True
2,adult_income,capital_gain,0.125391,age,0.085868,very_weak,True,True
3,adult_income,age,0.141710,hours_per_week,0.072244,very_weak,True,True
4,adult_income,education_num,0.171823,hours_per_week,0.067522,very_weak,True,True
5,adult_income,occupation,0.000000,None,0.065347,very_weak,False,True
6,adult_income,education,0.000000,None,0.065343,very_weak,False,True
7,adult_income,hours_per_week,0.171823,education_num,0.044944,very_weak,True,True
8,adult_income,capital_loss,0.066966,capital_gain,0.036241,very_weak,True,True
9,adult_income,sex,0.000000,None,0.027358,very_weak,False,True


In [ ]:
# ============================================================
# NOTEBOOK 05.20 — RESEARCH AUDIT SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.20")
print("RESEARCH AUDIT SUMMARY")
print("=" * 100)


TOTAL_TASKS = len(
    FEATURE_TASKS
)


TOTAL_COMPLETED = (

    len(
        SELECTED_STRATEGIES_DF
    )

)


TOTAL_FAILURES = len(
    FAILURES_DF
)


if TOTAL_TASKS > 0:

    COMPLETION_RATE = (

        TOTAL_COMPLETED
        /
        TOTAL_TASKS
        *
        100

    )

else:

    COMPLETION_RATE = 0.0


if not (
    RECOMMENDATION_VALIDATION_DF.empty
):

    VALIDATION_RATE = (

        RECOMMENDATION_VALIDATION_DF[
            "overall_valid"
        ].mean()
        *
        100

    )

else:

    VALIDATION_RATE = 0.0


AUDIT_SUMMARY = pd.DataFrame({

    "metric": [

        "LLM Provider",

        "LLM Model",

        "Device",

        "Total Feature Tasks",

        "Completed Features",

        "Failed Features",

        "Completion Rate (%)",

        "Recommendation Validation Rate (%)",

        "Deterministic Generation",

        "Master Seed"

    ],

    "value": [

        LLM_PROVIDER,

        LLM_MODEL,

        LLM_DEVICE,

        TOTAL_TASKS,

        TOTAL_COMPLETED,

        TOTAL_FAILURES,

        round(
            COMPLETION_RATE,
            2
        ),

        round(
            VALIDATION_RATE,
            2
        ),

        not LLM_DO_SAMPLE,

        LLM_SEED

    ]

})


display(
    AUDIT_SUMMARY
)


AUDIT_PATH = (

    RECOMMENDATION_DIR
    / "air_llm_audit_summary.csv"

)


AUDIT_SUMMARY.to_csv(

    AUDIT_PATH,

    index=False

)


print(
    f"\nAudit saved to:"
)

print(
    AUDIT_PATH
)


print("\n" + "=" * 100)
print("NOTEBOOK 05 AUDIT : COMPLETED")
print("=" * 100)

FEATURE PROFILE CONSTRUCTION
Datasets : 3
Features : 77
Complete profiles : 77
Incomplete profiles : 0


,dataset_id,feature,type_available,missingness_available,distribution_available,relevance_available,cardinality_available,dependency_available,complete_profile
0,adult_income,age,True,True,True,True,True,True,True
1,adult_income,workclass,True,True,True,True,True,True,True
2,adult_income,fnlwgt,True,True,True,True,True,True,True
3,adult_income,education,True,True,True,True,True,True,True
4,adult_income,education_num,True,True,True,True,True,True,True
5,adult_income,marital_status,True,True,True,True,True,True,True
6,adult_income,occupation,True,True,True,True,True,True,True
7,adult_income,relationship,True,True,True,True,True,True,True
8,adult_income,race,True,True,True,True,True,True,True
9,adult_income,sex,True,True,True,True,True,True,True



Feature profiles constructed using:
P_j = [T_j, M_j, D_j, R_j, C_j, Y_j]


In [ ]:
# ============================================================
# NOTEBOOK 05.21 — NOTEBOOK 05 MANIFEST
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.21")
print("NOTEBOOK 05 MANIFEST")
print("=" * 100)


def sha256_file(
    path
):

    import hashlib

    sha256 = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(

            lambda:
                f.read(
                    1024 * 1024
                ),

            b""

        ):

            sha256.update(
                chunk
            )


    return sha256.hexdigest()


MANIFEST_OUTPUTS = []


for path in [

    FULL_RECOMMENDATION_PATH,

    SELECTED_PATH,

    DISTRIBUTION_PATH,

    VALIDATION_PATH,

    AUDIT_PATH

]:

    if path.is_file():

        MANIFEST_OUTPUTS.append({

            "path":
                str(
                    path.relative_to(
                        PROJECT_ROOT
                    )
                ),

            "exists":
                True,

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(
                    path
                )

        })


NOTEBOOK_05_MANIFEST = {

    "project":
        "AIR-LLM",

    "notebook":
        "05_Local_LLM_Recommendation",

    "version":
        "2.0",

    "llm_provider":
        LLM_PROVIDER,

    "llm_model":
        LLM_MODEL,

    "device":
        LLM_DEVICE,

    "master_seed":
        LLM_SEED,

    "deterministic_generation":
        not LLM_DO_SAMPLE,

    "total_feature_tasks":
        int(
            TOTAL_TASKS
        ),

    "completed_features":
        int(
            TOTAL_COMPLETED
        ),

    "failed_features":
        int(
            TOTAL_FAILURES
        ),

    "completion_rate":
        float(
            COMPLETION_RATE
        ),

    "recommendation_validation_rate":
        float(
            VALIDATION_RATE
        ),

    "outputs":
        MANIFEST_OUTPUTS

}


MANIFEST_PATH = (

    RECOMMENDATION_DIR
    / "notebook_05_manifest.json"

)


with open(

    MANIFEST_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        NOTEBOOK_05_MANIFEST,

        f,

        indent=2

    )


print(
    f"Manifest saved:"
)

print(
    MANIFEST_PATH
)


print(
    f"Output artifacts: "
    f"{len(MANIFEST_OUTPUTS)}"
)


print("\n" + "=" * 100)
print("NOTEBOOK 05 MANIFEST : COMPLETED")
print("=" * 100)

NOTEBOOK 05 — SAVING PROFILE ARTIFACTS
Required profile objects: AVAILABLE

NOTEBOOK 05 — PROFILE ARTIFACTS SAVED
Dataset JSON : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/dataset/dataset_profiles.json
Dataset CSV  : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/dataset/dataset_profiles.csv
Feature JSON : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/feature/feature_profiles.json
Feature CSV  : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/feature/feature_profiles.csv
Registry     : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/profile_registry.csv

Dataset profiles : (3, 15)
Feature profiles : (77, 26)

NOTEBOOK 05 PROFILE SAVE COMPLETE


In [ ]:
# ============================================================
# NOTEBOOK 05.22 — FINAL NOTEBOOK CHECK
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 05.22")
print("FINAL NOTEBOOK 05 CHECK")
print("=" * 100)


CHECKS = {

    "TOKENIZER":
        TOKENIZER is not None,

    "LOCAL_LLM":
        LOCAL_LLM is not None,

    "CHAT_TEMPLATE":
        (
            hasattr(
                TOKENIZER,
                "apply_chat_template"
            )
            and
            TOKENIZER.chat_template is not None
        ),

    "GENERATION_ENGINE":
        callable(
            generate_local_response
        ),

    "JSON_EXTRACTION":
        callable(
            extract_json_object
        ),

    "VALIDATION_ENGINE":
        callable(
            validate_recommendation
        ),

    "RECOMMENDATION_RESULTS":
        (
            "AIR_LLM_RECOMMENDATIONS_DF"
            in globals()
        ),

    "SELECTED_STRATEGIES":
        (
            "SELECTED_STRATEGIES_DF"
            in globals()
        ),

    "AUDIT_SUMMARY":
        (
            "AUDIT_SUMMARY"
            in globals()
        ),

    "MANIFEST":
        MANIFEST_PATH.is_file()

}


for name, status in CHECKS.items():

    print(
        f"{name:<25} : "
        f"{'PASS' if status else 'FAIL'}"
    )


if not all(
    CHECKS.values()
):

    raise RuntimeError(
        "One or more Notebook 05 checks failed."
    )


print("\n" + "=" * 100)
print("NOTEBOOK 05 : COMPLETE AND VALID")
print("=" * 100)

print(
    "\nAIR-LLM local recommendation pipeline "
    "is ready for downstream evaluation."
)

AIR-LLM CONTEXT CONSTRUCTION COMPLETE
adult_income         | target: income          | features:  14
bank_marketing       | target: y               | features:  16
diabetes_130us       | target: readmitted      | features:  47

AIR-LLM context contains:
  • Dataset-level profile
  • Feature-level profiles
  • Target/task information

Context is ready for Notebook 06.
